# Vehicle Insurance Cross-Sell Response Prediction

## Life cycle of the machine learning project
- Understanding the problem statement
- Data collection
- Data checks to perform
- Exploratory data analysis
- Data pre-processing (decisions for the pipeline)
- Model training
- Choose best model

## Framing
1. **Business objective:** raise the share of contacted customers who buy vehicle
   insurance, within a limited outreach budget.
2. **ML problem type:** supervised binary classification, P(Response = 1 | x), with a
   cost-sensitive decision threshold.
3. **Baseline:** majority class (and random targeting). Any model must beat it on lift
   and PR-AUC, not accuracy.
4. **Constraints:** explainable (SHAP), fair across gender, simple to serve, and
   reproducible with one command.
5. **Metrics:** PR-AUC (primary), ROC-AUC, recall, top-10% lift, calibration.

## 1) Problem statement
This project predicts whether an existing life-insurance policyholder will respond
positively to a vehicle-insurance offer (`Response` = 1). The real output is a calibrated
probability, so a limited outreach budget can go to the customers most likely to say yes.

## 2) Data collection
- Source: Bank insurance analytics case study
- Files: train_data_cross-sell.csv (labelled), test_data_cross-sell.csv (no Response)
- Target: Response (1 = interested, 0 = not interested)
- **Size:** train has 324,466 rows and 12 columns; test has 56,643 rows and 11 columns

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

train = pd.read_csv("data/train_data_cross-sell.csv", dtype={"id": str})
test = pd.read_csv("data/test_data_cross-sell.csv", dtype={"id": str})
print(train.shape, test.shape)
train.head()

(324466, 12) (56643, 11)


,id,Gender,Customer_Date_of_Birth,Driving_License,Customer_Residence_Sub_County,Previously_Insured,Vehicle_Year_of_Manufacture,Vehicle_Damage,Annual_Premium,Agent_name,Life_policy_start_date,Response
0,549771831,Male,"Monday, March 7, 1977",1,EMBAKASI,0,2020,Yes,38294,Jomo Kenyatta,"Thursday, December 21, 2023",1
1,373583351,Male,"Sunday, December 29, 2002",1,WESTLANDS,1,2020,No,28619,Obiero Ochieng,"Wednesday, June 28, 2023",0
2,933163655,Female,"Tuesday, August 23, 1994",1,DAGORETTI,1,2011,No,27496,Obiero Ochieng,"Saturday, December 9, 2023",0
3,698942186,Female,"Thursday, November 18, 1999",1,JUJA,0,2020,Yes,2630,Gitonga Mwangi,"Tuesday, July 25, 2023",0
4,515109822,Male,"Thursday, February 22, 2001",1,WESTLANDS,0,2015,Yes,23367,Obiero Ochieng,"Saturday, May 13, 2023",0


In [5]:
print("Train rows, columns:", train.shape)
print("Test rows, columns:", test.shape)

Train rows, columns: (324466, 12)
Test rows, columns: (56643, 11)


## 2.2 Dataset information

- **id**: unique ID for the customer (read as text, so it is not corrupted into scientific notation)
- **Gender**: gender of the customer -> (Male/Female)
- **Customer_Date_of_Birth**: customer date of birth (text like "Tuesday, April 15, 1980", converted to a date later)
- **Driving_License**: whether the customer has a driving licence -> (0 = no, 1 = yes)
- **Customer_Residence_Sub_County**: sub-county where the customer lives (many unique values)
- **Previously_Insured**: whether the customer already has vehicle insurance -> (0 = no, 1 = yes)
- **Vehicle_Year_of_Manufacture**: year the vehicle was manufactured
- **Vehicle_Damage**: whether the customer's vehicle has been damaged in the past -> (Yes/No)
- **Annual_Premium**: the amount the customer pays as premium in the year
- **Agent_name**: name of the life insurance sales agent who sold the policy (many unique values)
- **Life_policy_start_date**: effective date of the customer's life insurance policy (text, converted to a date later)
- **Response** (target, train file only): whether the customer is interested in vehicle insurance -> (1 = interested, 0 = not interested)

## 3) Data checks to perform
- Check missing values
- Check duplicates
- Check data types
- Check the number of unique values in each column
- Check statistics of the data set
- Check the categories present in the categorical columns
- Check dates parse correctly and values are plausible (ages, vehicle years, premiums)

### 3.1 Check missing values

In [6]:
train.isna().sum()

id                               0
Gender                           0
Customer_Date_of_Birth           0
Driving_License                  0
Customer_Residence_Sub_County    0
Previously_Insured               0
Vehicle_Year_of_Manufacture      0
Vehicle_Damage                   0
Annual_Premium                   0
Agent_name                       0
Life_policy_start_date           0
Response                         0
dtype: int64

**Observation:** None of the 12 columns has a missing value (all counts are 0), so no imputation is needed for the train set.

### 3.2 Check duplicates

In [7]:
print("Duplicate rows:", train.duplicated().sum())
print("Duplicate ids:", train["id"].duplicated().sum())
print("Duplicates ignoring id:", train.drop(columns="id").duplicated().sum())

Duplicate rows: 0
Duplicate ids: 70
Duplicates ignoring id: 0


**Observation:**
- There are 0 fully duplicated rows, and 0 duplicates when `id` is ignored, so no customer record is repeated.
- However, 70 `id` values appear more than once (324,396 unique ids across 324,466 rows). Since the rows differ in every other column, these are not duplicate customers but repeated ids.
- This confirms that `id` cannot be trusted for deduplication or joins. It will be dropped from the model features and kept only to identify records.

### 3.3 Check data types

In [8]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 324466 entries, 0 to 324465
Data columns (total 12 columns):
 #   Column                         Non-Null Count   Dtype
---  ------                         --------------   -----
 0   id                             324466 non-null  str  
 1   Gender                         324466 non-null  str  
 2   Customer_Date_of_Birth         324466 non-null  str  
 3   Driving_License                324466 non-null  int64
 4   Customer_Residence_Sub_County  324466 non-null  str  
 5   Previously_Insured             324466 non-null  int64
 6   Vehicle_Year_of_Manufacture    324466 non-null  int64
 7   Vehicle_Damage                 324466 non-null  str  
 8   Annual_Premium                 324466 non-null  int64
 9   Agent_name                     324466 non-null  str  
 10  Life_policy_start_date         324466 non-null  str  
 11  Response                       324466 non-null  int64
dtypes: int64(5), str(7)
memory usage: 29.7 MB


**Observation:**
- 5 columns are integers (`Driving_License`, `Previously_Insured`, `Vehicle_Year_of_Manufacture`, `Annual_Premium`, `Response`) and 7 are text.
- `Customer_Date_of_Birth` and `Life_policy_start_date` are stored as long text strings and must be converted to dates (done in 3.7).
- `id` is kept as text so it is not turned into scientific notation.
- `Gender`, `Vehicle_Damage`, sub-county and agent are categorical and will be encoded in the pipeline.

### 3.4 Check the number of unique values of each column

In [9]:
train.nunique()

id                               324396
Gender                                2
Customer_Date_of_Birth            21108
Driving_License                       2
Customer_Residence_Sub_County        53
Previously_Insured                    2
Vehicle_Year_of_Manufacture          14
Vehicle_Damage                        2
Annual_Premium                    47027
Agent_name                          155
Life_policy_start_date              290
Response                              2
dtype: int64

**Observation:**
- `Gender`, `Driving_License`, `Previously_Insured`, `Vehicle_Damage` and `Response` are binary (2 values each).
- `Vehicle_Year_of_Manufacture` has only 14 values (2009 to 2022), so it behaves like a small ordered category.
- `Customer_Residence_Sub_County` has 53 values and `Agent_name` has 155. These are high-cardinality categoricals, so one-hot encoding would add many columns. They will be target-encoded inside cross-validation folds.
- `Life_policy_start_date` has 290 unique dates, which gives enough time points for a temporal split.
- `Annual_Premium` (47,027 values) and `Customer_Date_of_Birth` (21,108 values) are effectively continuous.

### 3.5 Check statistics of the data set

In [10]:
train.describe()

,Driving_License,Previously_Insured,Vehicle_Year_of_Manufacture,Annual_Premium,Response
count,324466.000000,324466.000000,324466.000000,324466.000000,324466.000000
mean,0.997938,0.457971,2015.048584,30570.897339,0.122805
std,0.045361,0.498231,3.745267,17196.763434,0.328214
min,0.000000,0.000000,2009.000000,2630.000000,0.000000
25%,1.000000,0.000000,2012.000000,24404.000000,0.000000
50%,1.000000,0.000000,2015.000000,31661.000000,0.000000
75%,1.000000,1.000000,2018.000000,39393.000000,0.000000
max,1.000000,1.000000,2022.000000,540165.000000,1.000000


**Observation:**
- `Response` has a mean of 0.123, so about 12.3% of customers are interested (the class imbalance is examined in section 4).
- `Driving_License` has a mean of 0.998, so almost everyone has a licence. It is nearly constant and will probably add little predictive value.
- `Previously_Insured` has a mean of 0.458, so about 46% already have vehicle insurance.
- `Vehicle_Year_of_Manufacture` runs from 2009 to 2022 (mean about 2015), which is plausible.
- `Annual_Premium` has a median of 31,661, and 75% of values are below 39,393, but the maximum is 540,165, more than 13 times the 75th percentile. These high outliers need a decision (cap, log-transform, or rely on tree models, which handle them well). The minimum is 2,630.

### 3.6 Check the categories in the categorical columns

In [11]:
low_card = ["Gender", "Driving_License", "Previously_Insured", "Vehicle_Damage", "Response"]
for col in low_card:
    print(col, "->", train[col].value_counts(dropna=False).to_dict())

for col in ["Customer_Residence_Sub_County", "Agent_name"]:
    print(f"\n{col}: {train[col].nunique()} unique")
    print(sorted(train[col].dropna().unique())[:60])

Gender -> {'Male': 175182, 'Female': 149284}
Driving_License -> {1: 323797, 0: 669}
Previously_Insured -> {0: 175870, 1: 148596}
Vehicle_Damage -> {'Yes': 163912, 'No': 160554}
Response -> {0: 284620, 1: 39846}

Customer_Residence_Sub_County: 53 unique
['ATHI RIVER', 'BONDO', 'BUMULA', 'BUNGOMA SOUTH', 'BURETI', 'DAGORETTI', 'EMBAKASI', 'IGEMBE CENTRAL', 'IMENTI SOUTH', 'ISINYA', 'JUJA', 'KABETE', 'KAJIADO NORTH', 'KAKAMEGA NORTH', 'KAMUKUNJI', 'KAPSERET', 'KASARANI', 'KIAMBAA', 'KILIFI SOUTH', 'KIMININI', 'KISAUNI', 'KISUMU EAST', 'KURIA WEST', 'KWANZA', "LANG'ATA", 'LIKONI', 'LUNGA LUNGA', 'MALINDI', 'MATHARE', 'NAIVASHA', 'NAKURU NORTH', 'NAKURU WEST', 'NAROK NORTH', 'NAROK SOUTH', 'NDHIWA', 'NJIRU', 'NJORO', 'NYALI', 'RONGAI', 'RUIRU', 'SAMBURU', 'SIAYA', 'SOTIK', 'SOY', 'STAREHE', 'THARAKA-NITHI', 'THIKA WEST', 'TRANS MARA WEST', 'TRANS NZOIA EAST', 'TRANS NZOIA WEST', 'TURBO', 'TURKANA WEST', 'WESTLANDS']

Agent_name: 155 unique
['Achieng Kwamboka', 'Akinyi Chepkorir', 'Akoth Che

**Observation:**
- `Gender`: Male 175,182 (54%) and Female 149,284 (46%), so the groups are reasonably balanced for the fairness check.
- `Driving_License`: 323,797 have a licence and only 669 (0.2%) do not.
- `Previously_Insured`: 175,870 not insured and 148,596 insured.
- `Vehicle_Damage`: Yes 163,912 and No 160,554, almost evenly split.
- `Response`: 284,620 not interested (0) and 39,846 interested (1), about 7 non-interested customers for every interested one. Accuracy would be misleading here, because predicting "not interested" for everyone scores about 87.7%. This supports using PR-AUC as the primary metric.
- Sub-county names (53) are all upper case and consistently spelled. The agent names that were printed (first 60 of 155) also look consistently formatted, with no obvious case or spacing differences.

### 3.7 Check that dates parse and values are plausible

In [12]:
fmt = "%A, %B %d, %Y"
for df in (train, test):
    df["Customer_Date_of_Birth"] = pd.to_datetime(df["Customer_Date_of_Birth"], format=fmt, errors="coerce")
    df["Life_policy_start_date"] = pd.to_datetime(df["Life_policy_start_date"], format=fmt, errors="coerce")
    df["age"] = (df["Life_policy_start_date"] - df["Customer_Date_of_Birth"]).dt.days / 365.25
    df["vehicle_age"] = df["Life_policy_start_date"].dt.year - df["Vehicle_Year_of_Manufacture"]

print("Unparsed dates:\n", train[["Customer_Date_of_Birth", "Life_policy_start_date"]].isna().sum(), "\n")
print("Policy start range:", train["Life_policy_start_date"].min(), "to", train["Life_policy_start_date"].max())
print("Age < 18:", (train["age"] < 18).sum(), "| Age > 90:", (train["age"] > 90).sum())
print("Negative vehicle age:", (train["vehicle_age"] < 0).sum(), "| Vehicle age > 40:", (train["vehicle_age"] > 40).sum())
print("Annual_Premium <= 0:", (train["Annual_Premium"] <= 0).sum())
train[["age", "vehicle_age", "Annual_Premium"]].describe()

Unparsed dates:
 Customer_Date_of_Birth    0
Life_policy_start_date    0
dtype: int64 

Policy start range: 2023-03-24 00:00:00 to 2024-01-07 00:00:00
Age < 18: 0 | Age > 90: 0
Negative vehicle age: 0 | Vehicle age > 40: 0
Annual_Premium <= 0: 0


,age,vehicle_age,Annual_Premium
count,324466.000000,324466.000000,324466.000000
mean,38.373297,7.975437,30570.897339
std,15.496602,3.748249,17196.763434
min,18.699521,1.000000,2630.000000
25%,24.156057,5.000000,24404.000000
50%,35.791923,8.000000,31661.000000
75%,48.943874,11.000000,39393.000000
max,84.895277,15.000000,540165.000000


**Observation:**
- All dates in both columns parsed successfully (0 unparsed).
- Life policies start between 24 March 2023 and 7 January 2024, about 9.5 months of data. A temporal split will therefore cover short windows (for example train on the earliest months, validate and test on the latest), and drift between them will be checked.
- Customer age at policy start ranges from 18.7 to 84.9 years (mean 38.4, median 35.8), so there are no minors or impossible ages.
- Vehicle age ranges from 1 to 15 years (mean 8.0) with no negative values.
- All annual premiums are positive. The only extreme values are the high outliers noted in 3.5.

### 3.8 Data quality note
The checks found no missing values, no duplicated rows, no unparsed dates and no
impossible ages or vehicle years, so this dataset is much cleaner than typical
real-world data. [Source of the explanation, e.g. "As explained in the case study /
by the data provider"], the data was anonymised and prepared before release, which
removed most obvious errors.

The checks still showed issues that the pipeline must handle:
- Dates stored as long text strings (parsed to dates, then converted to age and vehicle age)
- 70 repeated customer ids, so ids cannot be used for deduplication or joins
- High-cardinality categoricals (53 sub-counties, 155 agents) needing in-fold target encoding
- Class imbalance (12.3% positive), which rules out accuracy as a metric
- Strong right-skew and outliers in `Annual_Premium` (maximum 540,165)
- A near-constant `Driving_License` column (99.8% = 1)
- A short time window (about 9.5 months), which limits the temporal split